In [19]:
import json
import os

train_path = "dataset/squad/train-v1.1.json"
dev_path = "dataset/squad/dev-v1.1.json"

with open(train_path, "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open(dev_path, "r", encoding="utf-8") as f:
    dev_data = json.load(f)

print("Training articles:", len(train_data["data"]))
print("Validation articles:", len(dev_data["data"]))

Training articles: 442
Validation articles: 48


In [20]:
article = train_data["data"][0]
paragraph = article["paragraphs"][0]
qa = paragraph["qas"][0]

print("Title:", article["title"])

print("\nContext:")
print(paragraph["context"])

print("\nQuestion:")
print(qa["question"])

print("\nAnswer:")
print(qa["answers"][0]["text"])

print("\nAnswer Start Position:")
print(qa["answers"][0]["answer_start"])

Title: University_of_Notre_Dame

Context:
Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.

Question:
To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?

Answer:
Saint Bernadette Soubirous

Answer Start Position:
515


In [21]:
total_questions = 0

for article in train_data["data"]:
    for paragraph in article["paragraphs"]:
        total_questions += len(paragraph["qas"])

print("Total training questions:", total_questions)

Total training questions: 87599


In [22]:
samples = []

for article in train_data["data"]:
    for paragraph in article["paragraphs"]:
        for qa in paragraph["qas"]:
            samples.append({
                "context": paragraph["context"],
                "question": qa["question"],
                "answer": qa["answers"][0]["text"],
                "answer_start": qa["answers"][0]["answer_start"]
            })

print("Total samples:", len(samples))
print("\nFirst sample:")
print(samples[0])

Total samples: 87599

First sample:
{'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.', 'question': 'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?', 'answer': 'Saint Bernadette Soubirous', 'answer_start': 515}


In [23]:
contexts = [sample["context"] for sample in samples]
questions = [sample["question"] for sample in samples]
answers = [sample["answer"] for sample in samples]
answer_starts = [sample["answer_start"] for sample in samples]

print("Contexts:", len(contexts))
print("Questions:", len(questions))
print("Answers:", len(answers))
print("Answer positions:", len(answer_starts))

Contexts: 87599
Questions: 87599
Answers: 87599
Answer positions: 87599


In [24]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

print("Tokenizer loaded successfully!")

Tokenizer loaded successfully!


In [25]:
example = tokenizer(
    questions[0],
    contexts[0],
    truncation=True,
    max_length=384,
    padding="max_length"
)

print("Input IDs:", example["input_ids"][:20])
print("Attention Mask:", example["attention_mask"][:20])
print("Number of tokens:", len(example["input_ids"]))

Input IDs: [101, 2000, 3183, 2106, 1996, 6261, 2984, 9382, 3711, 1999, 8517, 1999, 10223, 26371, 2605, 1029, 102, 6549, 2135, 1010]
Attention Mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Number of tokens: 384


In [26]:
answer_text = answers[0]
answer_start = answer_starts[0]

answer_end = answer_start + len(answer_text)

print("Answer:", answer_text)
print("Character start:", answer_start)
print("Character end:", answer_end)

Answer: Saint Bernadette Soubirous
Character start: 515
Character end: 541


In [27]:
tokenized = tokenizer(
    questions[0],
    contexts[0],
    truncation=True,
    max_length=384,
    padding="max_length",
    return_offsets_mapping=True
)

offsets = tokenized["offset_mapping"]

start_token = None
end_token = None

for i, (start, end) in enumerate(offsets):
    if start <= answer_start < end:
        start_token = i

    if start < answer_end <= end:
        end_token = i

print("Answer:", answer_text)
print("Start token:", start_token)
print("End token:", end_token)

Answer: Saint Bernadette Soubirous
Start token: 130
End token: 137


In [28]:
tokenized_data = tokenizer(
    questions,
    contexts,
    truncation=True,
    max_length=384,
    padding="max_length"
)

print("Number of examples:", len(tokenized_data["input_ids"]))
print("Tokens per example:", len(tokenized_data["input_ids"][0]))

Number of examples: 87599
Tokens per example: 384


In [30]:
start_positions = []
end_positions = []

for i in range(len(samples)):
    answer_start = answer_starts[i]
    answer_end = answer_start + len(answers[i])

    offsets = tokenizer(
        questions[i],
        contexts[i],
        truncation=True,
        max_length=384,
        padding="max_length",
        return_offsets_mapping=True
    )["offset_mapping"]

    start_token = 0
    end_token = 0

    for token_index, (start, end) in enumerate(offsets):
        if start <= answer_start < end:
            start_token = token_index

        if start < answer_end <= end:
            end_token = token_index

    start_positions.append(start_token)
    end_positions.append(end_token)

print("Start positions:", start_positions[:5])
print("End positions:", end_positions[:5])

Start positions: [130, 52, 81, 95, 33]
End positions: [137, 56, 83, 101, 39]


In [31]:
import torch
from torch.utils.data import TensorDataset

input_ids = torch.tensor(tokenized_data["input_ids"])
attention_masks = torch.tensor(tokenized_data["attention_mask"])

start_positions_tensor = torch.tensor(start_positions)
end_positions_tensor = torch.tensor(end_positions)

train_dataset = TensorDataset(
    input_ids,
    attention_masks,
    start_positions_tensor,
    end_positions_tensor
)

print("Training examples:", len(train_dataset))

Training examples: 87599


In [32]:
from torch.utils.data import DataLoader

batch_size = 8

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

print("Number of batches:", len(train_loader))

Number of batches: 10950


In [33]:
from transformers import AutoModelForQuestionAnswering

model = AutoModelForQuestionAnswering.from_pretrained(
    "bert-base-uncased"
)

print("BERT Question Answering model loaded successfully!")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly i

BERT Question Answering model loaded successfully!


In [34]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

print("Training device:", device)

Training device: cpu


In [35]:
from torch.optim import AdamW

optimizer = AdamW(
    model.parameters(),
    lr=3e-5
)

print("Optimizer ready!")

Optimizer ready!


In [3]:
import json

train_path = "dataset/squad/train-v1.1.json"

with open(train_path, "r", encoding="utf-8") as f:
    train_data = json.load(f)

samples = []

for article in train_data["data"]:
    for paragraph in article["paragraphs"]:
        for qa in paragraph["qas"]:
            samples.append({
                "context": paragraph["context"],
                "question": qa["question"],
                "answer": qa["answers"][0]["text"],
                "answer_start": qa["answers"][0]["answer_start"]
            })

print("Total training samples:", len(samples))
print("First question:", samples[0]["question"])
print("First answer:", samples[0]["answer"])

Total training samples: 87599
First question: To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
First answer: Saint Bernadette Soubirous


In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print("Tokenizer loaded successfully!")
print("Fast tokenizer:", tokenizer.is_fast)

ImportError: DLL load failed while importing _regex: An Application Control policy has blocked this file.

In [4]:
from tqdm.auto import tqdm

start_positions = []
end_positions = []

batch_size = 500

for batch_start in tqdm(range(0, len(samples), batch_size)):
    batch_end = min(batch_start + batch_size, len(samples))
    batch_samples = samples[batch_start:batch_end]

    batch_questions = [s["question"] for s in batch_samples]
    batch_contexts = [s["context"] for s in batch_samples]

    encodings = tokenizer(
        batch_questions,
        batch_contexts,
        truncation="only_second",
        max_length=384,
        padding="max_length",
        return_offsets_mapping=True
    )

    for i, sample in enumerate(batch_samples):
        answer_start = sample["answer_start"]
        answer_end = answer_start + len(sample["answer"])

        offsets = encodings["offset_mapping"][i]
        sequence_ids = encodings.sequence_ids(i)

        start_token = 0
        end_token = 0

        context_token_indices = [
            j for j, seq_id in enumerate(sequence_ids)
            if seq_id == 1 and offsets[j][1] > offsets[j][0]
        ]

        if context_token_indices:
            first_context_token = context_token_indices[0]
            last_context_token = context_token_indices[-1]

            if (
                offsets[first_context_token][0] <= answer_start
                and offsets[last_context_token][1] >= answer_end
            ):
                for j in context_token_indices:
                    token_start, token_end = offsets[j]

                    if token_start <= answer_start < token_end:
                        start_token = j

                    if token_start < answer_end <= token_end:
                        end_token = j

        start_positions.append(start_token)
        end_positions.append(end_token)

print("Start positions:", start_positions[:5])
print("End positions:", end_positions[:5])
print("Total labels:", len(start_positions))
print("Total samples:", len(samples))

  0%|          | 0/176 [00:00<?, ?it/s]


NameError: name 'tokenizer' is not defined

In [38]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("PyTorch is currently using CPU.")

PyTorch version: 2.14.1+cpu
CUDA available: False
PyTorch is currently using CPU.


In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Not detected")

PyTorch: 2.14.1+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 5050 Laptop GPU


In [1]:

import sys
import torch
import transformers

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
print("Transformers:", transformers.__version__)


C:\Users\Shivam\Desktop\Question_Answering_WTP\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: 3.14.0
PyTorch: 2.14.1+cu130
GPU available: True
Transformers: 5.18.0


In [4]:

import json
from pathlib import Path

project_dir = Path.cwd()

train_path = project_dir / "dataset" / "squad" / "train-v1.1.json"
dev_path = project_dir / "dataset" / "squad" / "dev-v1.1.json"

print("Training file exists:", train_path.exists())
print("Validation file exists:", dev_path.exists())

with open(train_path, "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open(dev_path, "r", encoding="utf-8") as f:
    dev_data = json.load(f)

def extract_qa_examples(data):
    examples = []

    for article in data["data"]:
        for paragraph in article["paragraphs"]:
            context = paragraph["context"]

            for qa in paragraph["qas"]:
                # SQuAD v1.1 includes answerable questions.
                if not qa["answers"]:
                    continue

                answer = qa["answers"][0]
                answer_text = answer["text"]
                answer_start = answer["answer_start"]

                # Verify that the answer matches the context.
                if context[answer_start:answer_start + len(answer_text)] != answer_text:
                    continue

                examples.append({
                    "id": qa["id"],
                    "context": context,
                    "question": qa["question"],
                    "answer": answer_text,
                    "answer_start": answer_start
                })

    return examples

train_examples = extract_qa_examples(train_data)
dev_examples = extract_qa_examples(dev_data)

print("Valid training examples:", len(train_examples))
print("Valid validation examples:", len(dev_examples))
print("\nSample question:", train_examples[0]["question"])
print("Sample answer:", train_examples[0]["answer"])


Training file exists: True
Validation file exists: True
Valid training examples: 87599
Valid validation examples: 10570

Sample question: To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
Sample answer: Saint Bernadette Soubirous


In [5]:

from transformers import AutoTokenizer

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Start with a small sample to verify the preprocessing.
sample_examples = train_examples[:100]

questions = [item["question"] for item in sample_examples]
contexts = [item["context"] for item in sample_examples]

encoded = tokenizer(
    questions,
    contexts,
    max_length=384,
    truncation="only_second",
    stride=128,
    return_overflowing_tokens=True,
    return_offsets_mapping=True,
    padding="max_length"
)

print("Original examples:", len(sample_examples))
print("Tokenized features:", len(encoded["input_ids"]))
print("Tokens per feature:", len(encoded["input_ids"][0]))
print("First question:", sample_examples[0]["question"])
print("First answer:", sample_examples[0]["answer"])


C:\Users\Shivam\Desktop\Question_Answering_WTP\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Original examples: 100
Tokenized features: 100
Tokens per feature: 384
First question: To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
First answer: Saint Bernadette Soubirous


In [6]:

def map_answer_positions(examples, encoded):
    start_positions = []
    end_positions = []
    offset_mapping = encoded["offset_mapping"]
    sample_mapping = encoded["overflow_to_sample_mapping"]

    for feature_index in range(len(encoded["input_ids"])):
        sample_index = sample_mapping[feature_index]
        example = examples[sample_index]

        answer_start = example["answer_start"]
        answer_end = answer_start + len(example["answer"])

        offsets = offset_mapping[feature_index]
        sequence_ids = encoded.sequence_ids(feature_index)

        # Identify the tokens belonging to the passage, not the question.
        context_tokens = [
            i for i, seq_id in enumerate(sequence_ids)
            if seq_id == 1 and offsets[i] is not None
        ]

        if not context_tokens:
            start_positions.append(0)
            end_positions.append(0)
            continue

        first_token = context_tokens[0]
        last_token = context_tokens[-1]

        # If the answer is outside this passage window, use CLS as the label.
        if (
            offsets[first_token][0] > answer_start
            or offsets[last_token][1] < answer_end
        ):
            start_positions.append(0)
            end_positions.append(0)
            continue

        token_start = first_token
        while (
            token_start <= last_token
            and offsets[token_start][0] <= answer_start
        ):
            token_start += 1
        token_start -= 1

        token_end = last_token
        while (
            token_end >= first_token
            and offsets[token_end][1] >= answer_end
        ):
            token_end -= 1
        token_end += 1

        start_positions.append(token_start)
        end_positions.append(token_end)

    return start_positions, end_positions


start_positions, end_positions = map_answer_positions(
    sample_examples, encoded
)

print("Mapped features:", len(start_positions))
print("First answer token positions:", start_positions[0], end_positions[0])
print("First answer:", sample_examples[0]["answer"])


Mapped features: 100
First answer token positions: 130 137
First answer: Saint Bernadette Soubirous


In [7]:

def decode_answer_span(feature_index):
    start = start_positions[feature_index]
    end = end_positions[feature_index]

    if start == 0 and end == 0:
        return "[Answer not present in this window]"

    token_ids = encoded["input_ids"][feature_index][start:end + 1]
    return tokenizer.decode(token_ids, skip_special_tokens=True)


for i in range(min(5, len(sample_examples))):
    print("Question:", sample_examples[i]["question"])
    print("Expected:", sample_examples[i]["answer"])
    print("Mapped:  ", decode_answer_span(i))
    print("-" * 60)


Question: To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
Expected: Saint Bernadette Soubirous
Mapped:   saint bernadette soubirous
------------------------------------------------------------
Question: What is in front of the Notre Dame Main Building?
Expected: a copper statue of Christ
Mapped:   a copper statue of christ
------------------------------------------------------------
Question: The Basilica of the Sacred heart at Notre Dame is beside to which structure?
Expected: the Main Building
Mapped:   the main building
------------------------------------------------------------
Question: What is the Grotto at Notre Dame?
Expected: a Marian place of prayer and reflection
Mapped:   a marian place of prayer and reflection
------------------------------------------------------------
Question: What sits on top of the Main Building at Notre Dame?
Expected: a golden statue of the Virgin Mary
Mapped:   a golden statue of the virgin mary
----------------------------

In [8]:

from tqdm.auto import tqdm

MAX_LENGTH = 384
DOC_STRIDE = 128
BATCH_SIZE = 64

def prepare_features(examples, batch_size=64):
    all_input_ids = []
    all_attention_masks = []
    all_start_positions = []
    all_end_positions = []

    for batch_start in tqdm(
        range(0, len(examples), batch_size),
        desc="Preparing features"
    ):
        batch = examples[batch_start:batch_start + batch_size]

        encoded_batch = tokenizer(
            [item["question"] for item in batch],
            [item["context"] for item in batch],
            max_length=MAX_LENGTH,
            truncation="only_second",
            stride=DOC_STRIDE,
            return_overflowing_tokens=True,
            return_offsets_mapping=True,
            padding="max_length"
        )

        starts, ends = map_answer_positions(batch, encoded_batch)

        for i, input_ids in enumerate(encoded_batch["input_ids"]):
            all_input_ids.append(input_ids)
            all_attention_masks.append(encoded_batch["attention_mask"][i])
            all_start_positions.append(starts[i])
            all_end_positions.append(ends[i])

    return {
        "input_ids": all_input_ids,
        "attention_mask": all_attention_masks,
        "start_positions": all_start_positions,
        "end_positions": all_end_positions
    }

print("Feature preparation function created successfully.")
print("Maximum sequence length:", MAX_LENGTH)
print("Passage overlap:", DOC_STRIDE)
print("Batch size:", BATCH_SIZE)


Feature preparation function created successfully.
Maximum sequence length: 384
Passage overlap: 128
Batch size: 64


In [9]:

# Prepare a small subset first
train_subset = train_examples[:1000]
dev_subset = dev_examples[:200]

train_features = prepare_features(train_subset, batch_size=BATCH_SIZE)
dev_features = prepare_features(dev_subset, batch_size=BATCH_SIZE)

print("\nTraining examples:", len(train_subset))
print("Training features:", len(train_features["input_ids"]))
print("Validation examples:", len(dev_subset))
print("Validation features:", len(dev_features["input_ids"]))

print("\nTraining feature shapes:")
print("Input IDs:", len(train_features["input_ids"]), "x", len(train_features["input_ids"][0]))
print("Start labels:", len(train_features["start_positions"]))
print("End labels:", len(train_features["end_positions"]))


Preparing features: 100%|██████████| 4/4 [00:00<00:00, 13.48it/s]


Training examples: 1000
Training features: 1032
Validation examples: 200
Validation features: 200

Training feature shapes:
Input IDs: 1032 x 384
Start labels: 1032
End labels: 1032


In [10]:

import torch
from transformers import AutoModelForQuestionAnswering

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForQuestionAnswering.from_pretrained(model_name)
model.to(device)

print("Model loaded successfully!")
print("Model:", model_name)
print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory allocated (GB):",
          round(torch.cuda.memory_allocated(0) / (1024**3), 2))


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 408.51it/s]
[transformers] BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
qa_outputs.bias                            | MISSING    | 
qa_outputs.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not o

Model loaded successfully!
Model: bert-base-uncased
Device: cuda
GPU: NVIDIA GeForce RTX 5050 Laptop GPU
GPU memory allocated (GB): 0.41


In [11]:

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from tqdm.auto import tqdm

class QADataset(Dataset):
    def __init__(self, features):
        self.features = features

    def __len__(self):
        return len(self.features["input_ids"])

    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(
                self.features["input_ids"][idx], dtype=torch.long
            ),
            "attention_mask": torch.tensor(
                self.features["attention_mask"][idx], dtype=torch.long
            ),
            "start_positions": torch.tensor(
                self.features["start_positions"][idx], dtype=torch.long
            ),
            "end_positions": torch.tensor(
                self.features["end_positions"][idx], dtype=torch.long
            )
        }

train_dataset = QADataset(train_features)
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True
)

optimizer = AdamW(model.parameters(), lr=3e-5)

model.train()
total_loss = 0.0

for batch in tqdm(train_loader, desc="Training BERT"):
    batch = {key: value.to(device) for key, value in batch.items()}

    optimizer.zero_grad(set_to_none=True)

    outputs = model(**batch)
    loss = outputs.loss

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    total_loss += loss.item()

average_loss = total_loss / len(train_loader)

print("Training completed!")
print("Average training loss:", round(average_loss, 4))


Training BERT: 100%|██████████| 129/129 [00:49<00:00,  2.63it/s]

Training completed!
Average training loss: 3.9955


In [12]:

from pathlib import Path

for name in ["models", "results", "app"]:
    folder = Path.cwd() / name
    print(f"\n{name}/")

    if folder.exists():
        saved_files = [p for p in folder.rglob("*") if p.is_file()]
        if saved_files:
            for p in saved_files:
                print(f"  {p.relative_to(Path.cwd())}")
        else:
            print("  Empty")
    else:
        print("  Not created yet")





models/
  Empty

results/
  Empty

app/
  Empty


In [13]:

print("Model exists:", "model" in globals())
print("Tokenizer exists:", "tokenizer" in globals())
print("GPU available:", torch.cuda.is_available() if "torch" in globals() else "PyTorch not loaded")

if "model" in globals():
    print("Model device:", next(model.parameters()).device)
    print("Model is in training mode:", model.training)

Model exists: True
Tokenizer exists: True
GPU available: True
Model device: cuda:0
Model is in training mode: True


In [14]:

from pathlib import Path

model_dir = Path.cwd() / "models" / "bert_qa"
model_dir.mkdir(parents=True, exist_ok=True)

model.save_pretrained(model_dir)
tokenizer.save_pretrained(model_dir)

print("Model and tokenizer saved successfully!")
print("Saved location:", model_dir)
print("Files saved:")
for file in sorted(model_dir.iterdir()):
    if file.is_file():
        print(f" - {file.name} ({file.stat().st_size / (1024 * 1024):.2f} MB)")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.28it/s]

Model and tokenizer saved successfully!
Saved location: C:\Users\Shivam\Desktop\Question_Answering_WTP\models\bert_qa
Files saved:
 - config.json (0.00 MB)
 - model.safetensors (415.42 MB)
 - tokenizer.json (0.68 MB)
 - tokenizer_config.json (0.00 MB)


In [15]:

import json
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

project_dir = Path.cwd()
model_dir = project_dir / "models" / "bert_qa"
dev_path = project_dir / "dataset" / "squad" / "dev-v1.1.json"

# Load the saved model and tokenizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForQuestionAnswering.from_pretrained(model_dir).to(device)
model.eval()

# Reload validation examples
with open(dev_path, "r", encoding="utf-8") as f:
    dev_data = json.load(f)

def extract_qa_examples(data):
    examples = []

    for article in data["data"]:
        for paragraph in article["paragraphs"]:
            context = paragraph["context"]

            for qa in paragraph["qas"]:
                if not qa["answers"]:
                    continue

                answer = qa["answers"][0]
                answer_text = answer["text"]
                answer_start = answer["answer_start"]

                if context[answer_start:answer_start + len(answer_text)] == answer_text:
                    examples.append({
                        "id": qa["id"],
                        "context": context,
                        "question": qa["question"],
                        "answer": answer_text,
                        "answer_start": answer_start
                    })

    return examples

dev_examples = extract_qa_examples(dev_data)
dev_subset = dev_examples[:200]

print("Saved model loaded:", model_dir.exists())
print("Validation examples:", len(dev_examples))
print("Evaluation subset:", len(dev_subset))
print("Device:", device)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 385.14it/s]


Saved model loaded: True
Validation examples: 10570
Evaluation subset: 200
Device: cuda


In [16]:

import re
import string
import json
from collections import Counter
from tqdm.auto import tqdm

def normalize_answer(text):
    text = text.lower()
    text = "".join(ch for ch in text if ch not in string.punctuation)
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(text.split())

def exact_match_score(prediction, reference):
    return float(normalize_answer(prediction) == normalize_answer(reference))

def token_f1_score(prediction, reference):
    pred_tokens = normalize_answer(prediction).split()
    ref_tokens = normalize_answer(reference).split()

    common = Counter(pred_tokens) & Counter(ref_tokens)
    same = sum(common.values())

    if not pred_tokens or not ref_tokens:
        return float(pred_tokens == ref_tokens)
    if same == 0:
        return 0.0

    precision = same / len(pred_tokens)
    recall = same / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)

def predict_answer(question, context):
    inputs = tokenizer(
        question,
        context,
        max_length=384,
        truncation="only_second",
        return_offsets_mapping=True,
        return_tensors="pt"
    )

    offsets = inputs.pop("offset_mapping")[0].tolist()
    sequence_ids = tokenizer(
        question,
        context,
        max_length=384,
        truncation="only_second"
    ).sequence_ids()

    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    start_logits = outputs.start_logits[0]
    end_logits = outputs.end_logits[0]

    # Select a valid answer span from the context.
    best_score = float("-inf")
    best_start = best_end = None

    for start in range(len(offsets)):
        if sequence_ids[start] != 1:
            continue

        max_end = min(start + 30, len(offsets) - 1)

        for end in range(start, max_end + 1):
            if sequence_ids[end] != 1:
                break

            score = start_logits[start].item() + end_logits[end].item()
            if score > best_score:
                best_score = score
                best_start, best_end = start, end

    if best_start is None:
        return ""

    return context[offsets[best_start][0]:offsets[best_end][1]]

evaluation_examples = dev_subset[:200]
predictions = []
em_scores = []
f1_scores = []

for example in tqdm(evaluation_examples, desc="Evaluating BERT"):
    prediction = predict_answer(example["question"], example["context"])
    reference = example["answer"]

    predictions.append({
        "id": example["id"],
        "question": example["question"],
        "context": example["context"],
        "reference_answer": reference,
        "predicted_answer": prediction
    })

    em_scores.append(exact_match_score(prediction, reference))
    f1_scores.append(token_f1_score(prediction, reference))

em = 100 * sum(em_scores) / len(em_scores)
f1 = 100 * sum(f1_scores) / len(f1_scores)

results_dir = Path.cwd() / "results"
results_dir.mkdir(parents=True, exist_ok=True)

metrics = {
    "model": "bert-base-uncased fine-tuned on 1000 SQuAD training examples",
    "evaluation_examples": len(evaluation_examples),
    "exact_match_percent": round(em, 2),
    "token_f1_percent": round(f1, 2)
}

with open(results_dir / "evaluation_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

with open(results_dir / "predictions.json", "w", encoding="utf-8") as f:
    json.dump(predictions, f, indent=2, ensure_ascii=False)

print("\nEvaluation complete!")
print("Evaluated examples:", len(evaluation_examples))
print(f"Exact Match (EM): {em:.2f}%")
print(f"Token F1: {f1:.2f}%")
print("Results saved to:", results_dir)

Evaluating BERT: 100%|██████████| 200/200 [01:48<00:00,  1.83it/s]


Evaluation complete!
Evaluated examples: 200
Exact Match (EM): 19.50%
Token F1: 23.80%
Results saved to: C:\Users\Shivam\Desktop\Question_Answering_WTP\results


In [17]:

for i, item in enumerate(predictions[:5], start=1):
    print(f"\nExample {i}")
    print("Question:", item["question"])
    print("Correct answer:", item["reference_answer"])
    print("Predicted answer:", item["predicted_answer"])
    print("-" * 60)


Example 1
Question: Which NFL team represented the AFC at Super Bowl 50?
Correct answer: Denver Broncos
Predicted answer: Denver Broncos
------------------------------------------------------------

Example 2
Question: Which NFL team represented the NFC at Super Bowl 50?
Correct answer: Carolina Panthers
Predicted answer: Super Bowl 50
------------------------------------------------------------

Example 3
Question: Where did Super Bowl 50 take place?
Correct answer: Santa Clara, California
Predicted answer: Super Bowl 50
------------------------------------------------------------

Example 4
Question: Which NFL team won Super Bowl 50?
Correct answer: Denver Broncos
Predicted answer: Super Bowl 50
------------------------------------------------------------

Example 5
Question: What color was used to emphasize the 50th anniversary of the Super Bowl?
Correct answer: gold
Predicted answer: Super Bowl 50
------------------------------------------------------------


In [18]:
print("Model loaded:", "model" in globals())
print("Tokenizer loaded:", "tokenizer" in globals())
print("Training examples loaded:", "train_examples" in globals())
print("Training features loaded:", "train_features" in globals())
print("CUDA available:", torch.cuda.is_available())

Model loaded: True
Tokenizer loaded: True
Training examples loaded: True
Training features loaded: True
CUDA available: True


In [19]:

print("Training examples:", len(train_examples))
print("Training features:", len(train_features))
print("Model device:", next(model.parameters()).device)
print("Model training mode:", model.training)

Training examples: 87599
Training features: 4
Model device: cuda:0
Model training mode: False


In [20]:
print("Training feature type:", type(train_features))
print("Training feature count:", len(train_features))
print("Feature keys:", list(train_features.keys()) if isinstance(train_features, dict) else "Not a dictionary")

if isinstance(train_features, dict):
    for key, value in train_features.items():
        try:
            print(f"{key}: {len(value)} items")
        except TypeError:
            print(f"{key}: {type(value)}")

Training feature type: <class 'dict'>
Training feature count: 4
Feature keys: ['input_ids', 'attention_mask', 'start_positions', 'end_positions']
input_ids: 1032 items
attention_mask: 1032 items
start_positions: 1032 items
end_positions: 1032 items


In [21]:
print("prepare_features available:", "prepare_features" in globals())
print("map_answer_positions available:", "map_answer_positions" in globals())
print("MAX_LENGTH:", MAX_LENGTH if "MAX_LENGTH" in globals() else "Not defined")
print("DOC_STRIDE:", DOC_STRIDE if "DOC_STRIDE" in globals() else "Not defined")
print("Training feature count:", len(train_features["input_ids"]))

prepare_features available: True
map_answer_positions available: True
MAX_LENGTH: 384
DOC_STRIDE: 128
Training feature count: 1032


In [22]:

# Prepare features from the first 5,000 training examples
train_subset = train_examples[:5000]

train_features_new = prepare_features(train_subset)

print("Training examples selected:", len(train_subset))
print("Training features created:", len(train_features_new["input_ids"]))
print("Feature fields:", list(train_features_new.keys()))

Preparing features: 100%|██████████| 79/79 [00:04<00:00, 18.34it/s]

Training examples selected: 5000
Training features created: 5090
Feature fields: ['input_ids', 'attention_mask', 'start_positions', 'end_positions']


In [23]:

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from tqdm.auto import tqdm

class QADataset(Dataset):
    def __init__(self, features):
        self.features = features

    def __len__(self):
        return len(self.features["input_ids"])

    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(self.features["input_ids"][idx], dtype=torch.long),
            "attention_mask": torch.tensor(self.features["attention_mask"][idx], dtype=torch.long),
            "start_positions": torch.tensor(self.features["start_positions"][idx], dtype=torch.long),
            "end_positions": torch.tensor(self.features["end_positions"][idx], dtype=torch.long),
        }

# Use the newly prepared features
train_dataset = QADataset(train_features_new)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

# Continue training the current model
model.train()
optimizer = AdamW(model.parameters(), lr=2e-5)

total_loss = 0.0

for batch in tqdm(train_loader, desc="Training BERT"):
    batch = {key: value.to("cuda") for key, value in batch.items()}

    optimizer.zero_grad()
    outputs = model(**batch)
    loss = outputs.loss
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    total_loss += loss.item()

average_loss = total_loss / len(train_loader)

print(f"Training completed!")
print(f"Training features: {len(train_dataset)}")
print(f"Average training loss: {average_loss:.4f}")

Training BERT: 100%|██████████| 637/637 [03:55<00:00,  2.71it/s]

Training completed!
Training features: 5090
Average training loss: 1.9592


In [24]:
from pathlib import Path

model_dir = Path("models/bert_qa")
model_dir.mkdir(parents=True, exist_ok=True)

model.save_pretrained(model_dir)
tokenizer.save_pretrained(model_dir)

print("Improved model and tokenizer saved successfully!")
print("Location:", model_dir.resolve())

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.26it/s]

Improved model and tokenizer saved successfully!
Location: C:\Users\Shivam\Desktop\Question_Answering_WTP\models\bert_qa


In [25]:

model.eval()

correct_em = 0
total_f1 = 0.0
new_predictions = []

for item in tqdm(dev_subset, desc="Evaluating improved BERT"):
    predicted = predict_answer(item["question"], item["context"])
    reference = item["answer"]

    em = exact_match_score(predicted, reference)
    f1 = token_f1_score(predicted, reference)

    correct_em += em
    total_f1 += f1

    new_predictions.append({
        "question": item["question"],
        "reference_answer": reference,
        "predicted_answer": predicted,
        "exact_match": em,
        "f1": f1
    })

new_em = 100 * correct_em / len(dev_subset)
new_f1 = 100 * total_f1 / len(dev_subset)

print(f"Evaluation examples: {len(dev_subset)}")
print(f"Exact Match: {new_em:.2f}%")
print(f"Token F1: {new_f1:.2f}%")

Evaluating improved BERT: 100%|██████████| 200/200 [01:44<00:00,  1.92it/s]

Evaluation examples: 200
Exact Match: 60.50%
Token F1: 67.74%


In [26]:

import json
from pathlib import Path

results_dir = Path("results")
results_dir.mkdir(parents=True, exist_ok=True)

metrics = {
    "model": "BERT fine-tuned on 5000 SQuAD training examples",
    "evaluation_examples": len(dev_subset),
    "exact_match_percent": round(new_em, 2),
    "token_f1_percent": round(new_f1, 2)
}

with open(results_dir / "evaluation_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=4)

with open(results_dir / "predictions.json", "w", encoding="utf-8") as f:
    json.dump(new_predictions, f, indent=4, ensure_ascii=False)

print("Updated metrics and predictions saved successfully!")
print("Metrics:", metrics)

Updated metrics and predictions saved successfully!
Metrics: {'model': 'BERT fine-tuned on 5000 SQuAD training examples', 'evaluation_examples': 200, 'exact_match_percent': 60.5, 'token_f1_percent': 67.74}
